# Part 1 — Scope, Ownership, and Inputs

### Objective

Train three independent pooled dynamic model-family pipelines: Logistic Regression, Random Forest, and boosted trees.

### Inputs

- Versioned hourly snapshot dataset with inherited patient_id and split
- Fixed one-row-per-patient/selected-stay split manifest
- Feature dictionary and dataset provenance

Each family uses separate horizon-specific output heads for 6h, 12h, 24h, and 48h while sharing the same feature and split definitions.


# Part 2 — Configure Reproducible Experiments

### Objective

Define model runs before fitting.

### Required settings

- Dataset and label versions
- Model search spaces
- 6h, 12h, 24h, and 48h objectives
- Preprocessing and missingness rules
- Patient/snapshot weighting
- Internal grouped cross-validation
- Calibration options
- Random seeds and artifact paths

### Output

A run manifest linking data, code, configuration, software versions, and Git commit.


# Part 3 — Load Data and Enforce Split Integrity

### Objective

Load the modeling table and reconstruct partitions from the fixed patient-level manifest.

### Rules

- Never randomly split snapshot rows.
- Treat patient_id as the modeling grouping key and keep all snapshots and horizons from one patient in one split.
- Verify that each patient_id maps to exactly one selected icustay_id in this dataset version.
- Keep test outcomes unavailable to tuning and selection.
- Stop on patient overlap, schema mismatch, missing assignments, or duplicate snapshot keys.


# Part 4 — Define Features, Weighting, and Preprocessing

### Objective

Build preprocessing that is fitted on training patients and reused unchanged.

### Rules

- Include continuous hours_since_icu.
- Exclude identifiers, labels, onset times, future variables, and split columns.
- Fit imputers, scalers, encoders, and selectors without test information.
- Use patient-level or exposure-time weighting so long stays do not dominate.
- Preserve missingness and time-since-last-measurement when prespecified.
- Store exact feature order and units.


# Part 5 — Establish Baselines and Leakage Screens

### Objective

Verify the learning setup before fitting complex models.

### Checks

- Outcome prevalence by horizon
- Constant-risk and simple clinical baselines
- Missing and infinite values
- Probability bounds
- Feature-label alignment
- Duplicate or unchanged consecutive snapshots
- Suspicious predictors such as future interventions, full-stay summaries, or outcome-derived variables

All checks must pass before formal training.


# Part 6 — Train the Logistic Regression Family

### Objective

Fit pooled Logistic Regression using all eligible training snapshots, with horizon-specific heads when required.

### Rules

- Learn weights from historical training snapshots only.
- Use patient-grouped internal cross-validation for regularization.
- Include nonlinear time terms or prespecified interactions if needed for stable horizon behavior.
- Save preprocessing and all output heads as one family pipeline.

### Output

6h, 12h, 24h, and 48h validation risks from the Logistic Regression family.


# Part 7 — Train the Random Forest Family

### Objective

Fit pooled Random Forest estimators for the configured horizons.

### Rules

- Bound tree count, depth, leaf size, and feature sampling.
- Apply the same patient split and weighting design.
- Do not train separate models at the 6h, 12h, 24h, and 48h reporting checkpoints.
- Do not ensemble probabilities with another family.

### Output

6h, 12h, 24h, and 48h validation risks from the Random Forest family.


# Part 8 — Train the Boosted-Tree Family

### Objective

Fit XGBoost or LightGBM estimators for the configured horizons.

### Rules

- Select and version one implementation.
- Tune complexity, learning rate, estimators, and sampling without test information.
- Use early stopping only with an approved non-test set.
- Keep horizon outputs and calibration metadata explicit.

### Output

6h, 12h, 24h, and 48h validation risks from the boosted-tree family.


# Part 9 — Calibrate and Lock Candidate Pipelines

### Objective

Prepare reproducible candidate pipelines for validation-based comparison.

### Rules

- Evaluate calibration separately for each horizon.
- Fit any calibrator without test outcomes.
- Preserve calibrated and uncalibrated risk definitions.
- Check that shorter-horizon risk does not behave implausibly relative to longer-horizon risk.
- Do not select the operational model or alert threshold in this notebook.

### Output

Locked candidate versions and complete validation predictions.


# Part 10 — Generate Hourly and Event-Driven Risk Predictions

### Objective

Apply fixed model weights to hourly snapshots and actual AKI-relevant EHR update times.

### Clarification

Hourly snapshots define the core training grid. During event-driven inference, new EHR data update the current feature state and the same fixed pipeline recomputes risk; model weights are not retrained.

### Required outputs

Model family, horizon, patient/stay provenance identifiers, prediction time, hours_since_icu, risk, label eligibility, onset time, inherited split, and artifact versions.


# Part 11 — Extension: Predict Creatinine and Urine Trajectories

### Objective

After the classification MVP, predict future serum creatinine and urine-output distributions or trajectories at 6, 12, 24, and 48 hours.

### Design

- Train across the full eligible ICU timeline, including early snapshots.
- Support short or missing histories rather than delaying all prediction.
- Include current value, slope, extrema, variability, measurement recency, and relevant multimodal context.
- Consider a cold-start component if early performance is materially worse.

### Output

Future value distributions or quantiles with uncertainty, not only point estimates.


# Part 12 — Save Models and Export Predictions

### Objective

Save complete local model-family pipelines and immutable prediction artifacts.

### Required contents

Preprocessors, fitted horizon heads, calibrators, feature order, hyperparameters, weighting design, training metadata, dataset compatibility, and validation predictions.

### Rules

Do not commit model weights or patient-level predictions. Do not choose thresholds using test data.
